# SPARC Example 14: Outer Gap Distribution — All SPARC Galaxies

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The outer gap is V_adj(R2) - V_bary(R2) — the residual between the
omega-corrected velocity and baryonic velocity at the outermost ring.
A key result of Flynn (2026): all 84 outer gaps are negative,
meaning V_adj always falls below V_bary, ruling out dark-matter re-importation.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

results = []
for g in corpus['galaxies']:
    if g['survey'] != 'SPARC' or not g.get('data') or len(g['data']) < 3:
        continue
    d  = g['data']
    R  = [p['Rad']  for p in d]
    V  = [p['Vobs'] for p in d]
    R1, V1 = R[0],  V[0]
    R2, V2 = R[-1], V[-1]
    if R1<=0 or R2<=0 or V1<=0 or V2<=0:
        continue
    omega = (V2/R2 - V1/R1) * (R1/R2)**1.5
    V_adj = V2 - R2 * omega
    last  = d[-1]
    Vgas  = last.get('Vgas', 0)
    Vdisk = last.get('Vdisk', 0)
    Vbul  = last.get('Vbul', 0)
    Vbar_sq = Vgas**2 + Vdisk**2 + Vbul**2
    if Vbar_sq <= 0:
        continue
    Vbar = np.sqrt(Vbar_sq)
    gap  = V_adj - Vbar
    results.append({'galaxy': g['galaxy'], 'gap': gap,
                    'omega': omega, 'Vbar': Vbar})

gaps = [r['gap'] for r in results]
print(f"Galaxies with outer gap: {len(gaps)}")
print(f"All negative: {all(g < 0 for g in gaps)}")
print(f"Mean gap: {np.mean(gaps):.1f} km/s")
print(f"Std gap:  {np.std(gaps):.1f} km/s")
print(f"\nPublished (Flynn 2026): mean = -51.4 ± 25.0 km/s, all 84 negative")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(gaps, bins=25, color='#2166ac', alpha=0.8, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5,
           label='Gap = 0 boundary')
ax.axvline(np.mean(gaps), color='orange', linestyle='-', linewidth=1.5,
           label=f'Mean = {np.mean(gaps):.1f} km/s')
ax.set_xlabel(r'$V_{\rm adj}(R_2) - V_{\rm bar}(R_2)$ (km/s)', fontsize=11)
ax.set_ylabel('N galaxies', fontsize=11)
ax.set_title('Outer Gap Distribution — SPARC\n'
             'All negative: V_adj < V_bar at outermost ring (Flynn 2026)',
             fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ex14_outer_gap.png', dpi=150, bbox_inches='tight')
plt.show()